In [1]:
import os
from datasets import load_dataset

dataset_name = "SQuAD_2.0"

train_path = os.path.join(os.getcwd(), "Datasets", dataset_name, "train-00000-of-00001.parquet")
val_path = os.path.join(os.getcwd(), "Datasets", dataset_name, "validation-00000-of-00001.parquet")

dataset = load_dataset("parquet", data_files={'train': train_path, 'val': val_path})
dataset

DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 130319
    })
    val: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 11873
    })
})

In [2]:
from datasets import DatasetDict

# time to reduce the dataset
reduced_train_set = dataset["train"].shuffle(seed=42).select(range(65000))
reduced_val_set = dataset["val"].shuffle(seed=42).select(range(5500))
reduced_test_set = dataset["val"].shuffle(seed=42).select(range(5500, 6000))

reduced_set = DatasetDict({"train":reduced_train_set,
                           "val":reduced_val_set,
                           "test":reduced_test_set})

reduced_set

DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 65000
    })
    val: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 5500
    })
    test: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 500
    })
})

In [ ]:
def count_no_answer(examples):
    counter = 0
    for answer in examples["answers"]:
        if len(answer["answer_start"]) == 0:
            counter += 1

    return counter

In [ ]:
count_no_answer(reduced_train_set)

334

In [ ]:
count_no_answer(reduced_val_set)

2774

In [ ]:
count_no_answer(reduced_test_set)

2762

In [ ]:
from transformers import AutoTokenizer
model_name="distilbert/distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [ ]:
from processing import preprocess_train_function, preprocess_val_function

In [ ]:
# tokenized_train = reduced_set["train"].map(
#     preprocess_train_function,
#     batched=True,
#     remove_columns=reduced_set["train"].column_names,
# )

tokenized_train = dataset["train"].map(
    preprocess_train_function,
    batched=True,
    remove_columns=dataset["train"].column_names,
)
tokenized_train

Map:   0%|          | 0/130319 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask', 'start_positions', 'end_positions', 'example_id'],
    num_rows: 131754
})

In [ ]:
# tokenized_eval = reduced_set["val"].map(
#     preprocess_train_function,
#     batched=True,
#     remove_columns=reduced_set["val"].column_names,
# )

tokenized_eval = dataset["val"].map(
    preprocess_train_function,
    batched=True,
    remove_columns=dataset["val"].column_names,
)
tokenized_eval

Map:   0%|          | 0/11873 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask', 'start_positions', 'end_positions', 'example_id'],
    num_rows: 12134
})

In [ ]:
import collections

val_example_to_features = collections.defaultdict(list)
for idx, feature in enumerate(tokenized_eval):
    val_example_to_features[feature["example_id"]].append(idx)

len(val_example_to_features)

650

In [ ]:
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback
from transformers import AutoModelForQuestionAnswering
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = AutoModelForQuestionAnswering.from_pretrained(model_name,
                                                      ).to(device)

output_name = "tt_bert"
output_dir = os.path.join(os.getcwd(), "models", output_name)

training_args = TrainingArguments(
    output_dir=output_dir,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    learning_rate=5e-6,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    push_to_hub=False,
    load_best_model_at_end=True,
    fp16=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    processing_class=tokenizer,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2, early_stopping_threshold=0.01)],
)

Some weights of DistilBertForQuestionAnswering were not initialized from the model checkpoint at distilbert/distilbert-base-uncased and are newly initialized: ['qa_outputs.bias', 'qa_outputs.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,2.049100,1.449323
2,1.430000,1.364823
3,1.293100,1.351313


TrainOutput(global_step=24705, training_loss=1.590741215088039, metrics={'train_runtime': 4108.766, 'train_samples_per_second': 96.2, 'train_steps_per_second': 6.013, 'total_flos': 3.873165421863629e+16, 'train_loss': 1.590741215088039, 'epoch': 3.0})

In [3]:
max_length = 384
stride = 128

def preprocess_validation_examples(examples):
    questions = [q.strip() for q in examples["question"]]
    inputs = tokenizer(
        questions,
        examples["context"],
        max_length=max_length,
        truncation="only_second",
        stride=stride,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )

    sample_map = inputs.pop("overflow_to_sample_mapping")
    example_ids = []

    for i in range(len(inputs["input_ids"])):
        sample_idx = sample_map[i]
        example_ids.append(examples["id"][sample_idx])

        sequence_ids = inputs.sequence_ids(i)
        offset = inputs["offset_mapping"][i]
        inputs["offset_mapping"][i] = [
            o if sequence_ids[k] == 1 else None for k, o in enumerate(offset)
        ]

    inputs["example_id"] = example_ids
    return inputs

In [4]:
import os
from transformers import AutoTokenizer

output_name = "tt_bert"
output_dir = os.path.join(os.getcwd(), "models", output_name, "checkpoint-24705")

tokenizer = AutoTokenizer.from_pretrained(output_dir)

tokenized_test = reduced_set["test"].map(
    preprocess_validation_examples,
    batched=True,
    remove_columns=reduced_set["test"].column_names,
)

# tokenized_test = dataset["val"].map(
#     preprocess_validation_examples,
#     batched=True,
#     remove_columns=dataset["val"].column_names,
# )
tokenized_test

Dataset({
    features: ['input_ids', 'attention_mask', 'offset_mapping', 'example_id'],
    num_rows: 509
})

In [5]:
import torch
from transformers import AutoModelForQuestionAnswering

t_tokenized_test = tokenized_test.remove_columns(["example_id", "offset_mapping"])
t_tokenized_test.set_format("torch")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
trained_model = AutoModelForQuestionAnswering.from_pretrained(output_dir).to(device)

batch = {k: t_tokenized_test[k].to(device) for k in t_tokenized_test.column_names}

with torch.no_grad():
    outputs = trained_model(**batch)

In [6]:
from torch.utils.data import DataLoader
import torch
from transformers import AutoModelForQuestionAnswering
import torch.nn.functional as F

t_tokenized_test = tokenized_test.remove_columns(["example_id", "offset_mapping"])
t_tokenized_test.set_format("torch")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
trained_model = AutoModelForQuestionAnswering.from_pretrained(output_dir).to(device)

batch_size = 64  # Adjust this based on your GPU memory
test_dataloader = DataLoader(t_tokenized_test, batch_size=batch_size)

start_probs = []
end_probs = []

trained_model.eval()  # Set the model to evaluation mode
with torch.no_grad():
    for batch in test_dataloader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = trained_model(**batch)
        start_logits = F.softmax(outputs.start_logits, dim=-1).cpu().numpy()
        end_logits = F.softmax(outputs.end_logits, dim=-1).cpu().numpy()

        start_probs.extend(start_logits)
        end_probs.extend(end_logits)

# Now all_start_logits and all_end_logits contain the predictions for your entire test set

In [6]:
import torch.nn.functional as F

start_probs = F.softmax(outputs.start_logits, dim=-1).cpu().numpy()
end_probs = F.softmax(outputs.end_logits, dim=-1).cpu().numpy()

In [7]:
import collections

example_to_features = collections.defaultdict(list)
for idx, feature in enumerate(tokenized_test):
    example_to_features[feature["example_id"]].append(idx)

In [8]:
# small_eval_set = reduced_set["test"]
# eval_Set = tokenised_test
import numpy as np
from tqdm import tqdm

n_best = 5
max_answer_length = 30
predicted_answers = []
no_answer_threshold = 0.8

for example in tqdm(reduced_set["test"]):
    example_id = example["id"]
    context = example["context"]
    answers = []    # track all answers for this id

    # and each feature, if the context was truncated
    for feature_index in example_to_features[example_id]:
        start_prob = start_probs[feature_index]
        end_prob = end_probs[feature_index]
        offsets = tokenized_test["offset_mapping"][feature_index]

        # get the n highest logit values
        start_indexes = np.argsort(start_prob)[-1 : -n_best - 1 : -1].tolist()
        end_indexes = np.argsort(end_prob)[-1 : -n_best - 1 : -1].tolist()

        # no answer is probability that CLS is start and end token
        no_answer_probability = start_prob[0] * end_prob[0]

        for start_index in start_indexes:
            for end_index in end_indexes:
                # Skip answers that are not fully in the context
                if offsets[start_index] is None or offsets[end_index] is None:
                    continue
                # Skip answers with a length that is either < 0 or > max_answer_length.
                if (end_index < start_index or end_index - start_index + 1 > max_answer_length
                ):
                    continue

                answers.append(
                    {
                        "text": context[offsets[start_index][0] : offsets[end_index][1]],
                        "prob_score": start_prob[start_index] * end_prob[end_index],
                        "no_answer_probability": no_answer_probability,
                    }
                )
    
    if len(answers):
        # check for the best answer for each id
        best_answer = max(answers, key=lambda x: x["prob_score"])
        predicted_answers.append({"id": example_id, "prediction_text": best_answer["text"], "no_answer_probability": best_answer["no_answer_probability"]})
    else:
        predicted_answers.append({"id": example_id, "prediction_text": "", "no_answer_probability": 1.0})

100%|██████████| 500/500 [03:39<00:00,  2.27it/s]


In [ ]:
import numpy as np

n_best = 20
max_answer_length = 30
predicted_answers_2 = []

for example in reduced_set["test"]:
    example_id = example["id"]
    context = example["context"]
    answers = []    # track all answers for this id

    for feature_index in example_to_features[example_id]:
        start_prob = start_probs[feature_index]
        end_prob = end_probs[feature_index]
        offsets = tokenized_test["offset_mapping"][feature_index]

        start_indexes = np.argsort(start_prob)[-1 : -n_best - 1 : -1]
        end_indexes = np.argsort(end_prob)[-1 : -n_best - 1 : -1]

        no_answer_probability = start_prob[0] * end_prob[0]

        # filter out values that equal none
        valid_start_indexes = [idx for idx in start_indexes if offsets[idx] is not None]
        valid_end_indexes = [idx for idx in end_indexes if offsets[idx] is not None]

        if not valid_start_indexes or not valid_end_indexes:
            continue  # Skip if no valid start or end indices

        start_indices_grid, end_indices_grid = np.meshgrid(valid_start_indexes, valid_end_indexes)
        flat_start_indices = start_indices_grid.flatten()
        flat_end_indices = end_indices_grid.flatten()

        # Filter by length
        length_mask = np.logical_and(
            flat_end_indices >= flat_start_indices,
            flat_end_indices - flat_start_indices + 1 <= max_answer_length,
        )

        final_start_indices = flat_start_indices[length_mask]
        final_end_indices = flat_end_indices[length_mask]

        # Calculate probability scores
        final_prob_scores = start_prob[final_start_indices] * end_prob[final_end_indices]

        max_idx = np.argmax(final_prob_scores)

        answers.append(
            {
                "text": context[offsets[final_start_indices[max_idx]][0] : offsets[final_end_indices[max_idx]][1]],
                "score": final_prob_scores[max_idx],
                "no_answer_probability": no_answer_probability,
            }
        )

    # check for the best answer for each id
    best_answer = max(answers, key=lambda x: x["score"])
    predicted_answers_2.append({"id": example_id, "prediction_text": best_answer["text"], "no_answer_probability": best_answer["no_answer_probability"]})

In [9]:
references = [{"id": ex["id"], "answers": ex["answers"]} for ex in reduced_set["test"]]

In [10]:
import evaluate
metric = evaluate.load("squad_v2")
metric.compute(predictions=predicted_answers, references=references)

{'exact': 36.0,
 'f1': 38.906384716961966,
 'total': 500,
 'HasAns_exact': 60.49382716049383,
 'HasAns_f1': 66.47404262749376,
 'HasAns_total': 243,
 'NoAns_exact': 12.84046692607004,
 'NoAns_f1': 12.84046692607004,
 'NoAns_total': 257,
 'best_exact': 55.8,
 'best_exact_thresh': 1.9080599600318493e-10,
 'best_f1': 56.21142857142857,
 'best_f1_thresh': 1.9080599600318493e-10}

In [ ]:
import evaluate
from torch.nn.functional import softmax
import torch

metric = evaluate.load("squad_v2")
# needs to be fixed group data into collections and other bs
def compute_metrics(p):
    predictions = p.predictions
    label_ids = p.label_ids

    # Convert to list of answers
    formatted_predictions = []
    formatted_references = []
    
    for idx, (pred, ref) in enumerate(zip(predictions, label_ids)):
        # Get start and end logits and no_answer probability
        start_logits = pred[0]
        end_logits = pred[1]
        no_answer_prob = pred[2]  # no_answer probability from model
        
        # Extract the span of the answer
        start_idx = torch.argmax(torch.tensor(start_logits)).item()
        end_idx = torch.argmax(torch.tensor(end_logits)).item()
        prediction_text = tokenizer.decode(
            tokenized_eval["input_ids"][start_idx:end_idx+1],
            skip_special_tokens=True
        )

        formatted_predictions.append({
            "id": tokenized_eval["example_id"],
            "prediction_text": prediction_text,
            "no_answer_probability": no_answer_prob
        })
        
        formatted_references.append({
            "id": tokenized_eval["example_id"],
            "answers": dataset["validation"][idx]["answers"]
        })
    
    result = metric.compute(predictions=formatted_predictions, references=formatted_references)
    
    return {
        "exact_match": result["exact"],
        "f1": result["f1"]
    }
